# EMA Cross Backtest — EUR/USD April 2024

**What this notebook does**

- Loads 1-minute **BID and ASK** OHLCV CSVs for EUR/USD April 2024 from `C:\Users\HP\Desktop\MS\Dataset\FX_yyyy\Fx\EUROUSD\2024\04`.
- Feeds both sides into a Nautilus `BacktestEngine` as **1-MINUTE-{BID|ASK}-EXTERNAL** bars. The matching engine maintains an L1 book; without both sides, market orders can't cross and end up rejected.
- The `EMACrossStrategy` subscribes to **5-MINUTE-BID-INTERNAL@1-MINUTE-EXTERNAL** so Nautilus's built-in `TimeBarAggregator` produces 5-min bars from the 1-min EXTERNAL feed automatically.
- After the run, writes **five CSV reports** to `reports/ema_cross_april2024/`:
    1. `order_fills_report.csv` — one row per filled order
    2. `positions_report.csv` — one row per position (opened/closed timestamps, Nautilus's built-in wide view)
    3. `account_report.csv` — account state snapshots
    4. `order_lifecycle_report.csv` — **one row per `OrderEvent`** with `ts_event` + `ts_init` + reason for every transition: `OrderInitialized → OrderSubmitted → OrderAccepted → (OrderTriggered) → OrderFilled` plus any `Rejected`/`Canceled`.
    5. `position_lifecycle_report.csv` — **one row per position-state transition** (`PositionOpened` / `PositionClosed`) with the same UTC + IST timestamp treatment. Links back to the order CSV via `opening_order_id` / `closing_order_id`.

> ⚠ **Backtest caveat for lifecycle timestamps:** Nautilus's `SimulatedExchange` processes submit → accept → fill within the same engine tick — the matching engine processes a bar's data into its L1 book *before* `on_bar()` fires, so a market order submitted in `on_bar(T)` fills against that bar and stamps at T. All events of one order share the bar-close timestamp. This is the correct default Nautilus behaviour, not a bug. If you ever want "next-bar fill" semantics, add `LatencyModel(insert_latency_nanos=60_000_000_000)` to the venue.

## 1. Imports & path setup

In [1]:
from __future__ import annotations

import sys
from decimal import Decimal
from pathlib import Path

import pandas as pd

# Make the project root importable so `core.*` and `strategies.*` resolve.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "ipynb" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.config import BacktestEngineConfig, LoggingConfig, RiskEngineConfig
from nautilus_trader.model.currencies import USD
from nautilus_trader.model.data import BarType
from nautilus_trader.model.enums import AccountType, OmsType
from nautilus_trader.model.identifiers import TraderId
from nautilus_trader.model.objects import Money
from nautilus_trader.persistence.wranglers import BarDataWrangler

from core.instrument_factory import create_instrument
from strategies.ema_cross import EMACrossConfig, EMACrossStrategy

print(f"Project root: {PROJECT_ROOT}")

Project root: d:\mcube_test_version_1_updated\m-cube_version1


## 2. Configuration

In [2]:
DATA_ROOT = Path(r"C:\Users\HP\Desktop\MS\Dataset\FX_yyyy\Fx\EUROUSD\2024\04")
OUT_DIR   = PROJECT_ROOT / "reports" / "ema_cross_april2024"

BASE, QUOTE, VENUE_NAME = "EUR", "USD", "FOREX_MS"
STRATEGY_SIDE = "BID"              # side the EMA strategy uses for signals
FAST_EMA      = 10
SLOW_EMA      = 20
TRADE_SIZE    = Decimal("100000")  # one standard FX lot
STARTING_USD  = 1_000_000

# Passive-limit entry offset.  EUR/USD price_increment = 0.00001, so:
#   50 ticks = 5 pips  → fills only when price moves ~5 pips against the signal
#                        (limit BUY rests BELOW close; limit SELL rests ABOVE close).
# Set to 0 to fall back to MARKET orders (legacy behaviour).
LIMIT_OFFSET_TICKS = 50

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data root:           {DATA_ROOT}")
print(f"Output dir:          {OUT_DIR}")
print(f"Limit offset ticks:  {LIMIT_OFFSET_TICKS}  (0 = MARKET orders, >0 = passive LIMIT)")

Data root:           C:\Users\HP\Desktop\MS\Dataset\FX_yyyy\Fx\EUROUSD\2024\04
Output dir:          d:\mcube_test_version_1_updated\m-cube_version1\reports\ema_cross_april2024
Limit offset ticks:  50  (0 = MARKET orders, >0 = passive LIMIT)


## 3. Load all 1-minute CSVs for April 2024 — BID and ASK

The data directory contains one subdir per trading day (`01/`, `02/`, …, `30/`), each holding `DD.04.2024_BID_OHLCV.csv` and `DD.04.2024_ASK_OHLCV.csv`. Schema: `timestamp, open, high, low, close, volume` with `timestamp` formatted as `DD.MM.YYYY HH:MM:SS.fff GMT+HHMM`.

In [3]:
def load_1min_ohlcv(data_root: Path, side: str) -> pd.DataFrame:
    """Concatenate every DD/{DD.MM.YYYY}_{side}_OHLCV.csv under data_root."""
    frames: list[pd.DataFrame] = []
    for day_dir in sorted(p for p in data_root.iterdir() if p.is_dir()):
        for csv_path in day_dir.glob(f"*_{side}_OHLCV.csv"):
            df = pd.read_csv(csv_path)
            df["timestamp"] = pd.to_datetime(
                df["timestamp"],
                format="%d.%m.%Y %H:%M:%S.%f GMT%z",
                utc=True,
            )
            frames.append(df)
    if not frames:
        raise FileNotFoundError(f"No *_{side}_OHLCV.csv files under {data_root}")
    full = pd.concat(frames, ignore_index=True)
    full = full.set_index("timestamp").sort_index()
    full = full[~full.index.duplicated(keep="first")]
    return full[["open", "high", "low", "close", "volume"]]

df_bid = load_1min_ohlcv(DATA_ROOT, "BID")
df_ask = load_1min_ohlcv(DATA_ROOT, "ASK")
print(f"BID: {len(df_bid):,} rows  ({df_bid.index.min()} -> {df_bid.index.max()})")
print(f"ASK: {len(df_ask):,} rows  ({df_ask.index.min()} -> {df_ask.index.max()})")
df_bid.head()

BID: 31,530 rows  (2024-03-31 21:00:00+00:00 -> 2024-04-30 18:29:00+00:00)
ASK: 31,530 rows  (2024-03-31 21:00:00+00:00 -> 2024-04-30 18:29:00+00:00)


,open,high,low,close,volume
timestamp,,,,,
2024-03-31 21:00:00+00:00,1.07882,1.07882,1.07848,1.07871,5.40
2024-03-31 21:01:00+00:00,1.07871,1.07871,1.07871,1.07871,0.75
2024-03-31 21:02:00+00:00,1.07876,1.07889,1.07871,1.07888,11.40
2024-03-31 21:03:00+00:00,1.07890,1.07890,1.07890,1.07890,0.90
2024-03-31 21:04:00+00:00,1.07888,1.07888,1.07888,1.07888,0.90


## 4. Wrangle into Nautilus `Bar` objects (both sides)

Each side gets its own EXTERNAL bar type. Nautilus internally aggregates the strategy side into 5-min INTERNAL bars when the strategy subscribes via the composite `5-MINUTE-{SIDE}-INTERNAL@1-MINUTE-EXTERNAL`. The other side's 1-min bars feed the matching engine so the L1 book has both bid and ask available for fills.

In [4]:
instrument = create_instrument(BASE, QUOTE, venue=VENUE_NAME)
venue = instrument.id.venue

bar_type_1m_bid = BarType.from_str(f"{instrument.id}-1-MINUTE-BID-EXTERNAL")
bar_type_1m_ask = BarType.from_str(f"{instrument.id}-1-MINUTE-ASK-EXTERNAL")
bar_type_5m_strategy = BarType.from_str(
    f"{instrument.id}-5-MINUTE-{STRATEGY_SIDE}-INTERNAL@1-MINUTE-EXTERNAL"
)

bars_bid = BarDataWrangler(bar_type_1m_bid, instrument).process(df_bid)
bars_ask = BarDataWrangler(bar_type_1m_ask, instrument).process(df_ask)

print(f"Instrument:       {instrument.id}")
print(f"BID feed:         {bar_type_1m_bid}  ({len(bars_bid):,} bars)")
print(f"ASK feed:         {bar_type_1m_ask}  ({len(bars_ask):,} bars)")
print(f"Strategy bartype: {bar_type_5m_strategy}")

Instrument:       EURUSD.FOREX_MS
BID feed:         EURUSD.FOREX_MS-1-MINUTE-BID-EXTERNAL  (31,530 bars)
ASK feed:         EURUSD.FOREX_MS-1-MINUTE-ASK-EXTERNAL  (31,530 bars)
Strategy bartype: EURUSD.FOREX_MS-5-MINUTE-BID-INTERNAL@1-MINUTE-EXTERNAL


## 5. Build the BacktestEngine and add venue / instrument / data / strategy

In [5]:
from nautilus_trader.model.enums import OrderSide, TimeInForce


class EMACrossLimitConfig(EMACrossConfig, frozen=True):
    """Same fields as EMACrossConfig + a `limit_offset_ticks` knob.

    When > 0 the strategy submits passive LIMIT entries priced
    `limit_offset_ticks * instrument.price_increment` away from the
    bar's close (BELOW close for BUY, ABOVE close for SELL) so the
    order rests until the market revisits the price. When 0 the
    strategy falls back to the parent's MARKET-order behaviour.
    """
    limit_offset_ticks: int = 0


class EMACrossLimitStrategy(EMACrossStrategy):
    """Subclass overriding entry submission to use LIMIT orders.

    On every fresh bar we first cancel any resting limit from a prior
    signal (so flips don't leave stale orders piling up). Exits via
    `close_all_positions()` keep using market orders — they shouldn't
    wait.
    """

    def __init__(self, config: EMACrossLimitConfig) -> None:
        super().__init__(config)
        self._last_bar = None  # cached so _submit_order can read bar.close

    def on_bar(self, bar) -> None:
        # Same filter / warmup gate as parent, then cancel pending limits.
        if bar.bar_type.standard() != self.config.bar_type.standard():
            return
        if not self.indicators_initialized():
            return
        self._last_bar = bar
        if self.cache.orders_open_count(instrument_id=self.config.instrument_id) > 0:
            self.cancel_all_orders(self.config.instrument_id)
        # Delegate the actual trading decision to the parent — it will
        # call our overridden _submit_order below for entries.
        super().on_bar(bar)

    def _submit_order(self, side: OrderSide) -> None:
        # Fall back to market when the offset is disabled (or no bar yet).
        if self.config.limit_offset_ticks <= 0 or self._last_bar is None:
            return super()._submit_order(side)

        fp = int(self.config.fast_ema_period)
        sp = int(self.config.slow_ema_period)
        fv = self.fast_ema.value
        sv = self.slow_ema.value
        side_label = "BUY" if side == OrderSide.BUY else "SELL"
        reason = (
            f"EMA Cross LIMIT {side_label}: fast({fp})={fv:.5f}  slow({sp})={sv:.5f}"
        )

        tick = float(self.instrument.price_increment)
        offset = self.config.limit_offset_ticks * tick
        close = float(self._last_bar.close)
        if side == OrderSide.BUY:
            limit_px = self.instrument.make_price(close - offset)  # passive: wait for dip
        else:
            limit_px = self.instrument.make_price(close + offset)  # passive: wait for rise

        order = self.order_factory.limit(
            instrument_id=self.config.instrument_id,
            order_side=side,
            quantity=self.instrument.make_qty(self.config.trade_size),
            price=limit_px,
            time_in_force=TimeInForce.GTC,
            tags=[reason],
        )
        self.submit_order(order)


print("Defined EMACrossLimitStrategy / EMACrossLimitConfig (inline subclass).")

Defined EMACrossLimitStrategy / EMACrossLimitConfig (inline subclass).


In [6]:
engine = BacktestEngine(
    config=BacktestEngineConfig(
        trader_id=TraderId("EMA-CROSS-APR2024"),
        logging=LoggingConfig(bypass_logging=True),
        risk_engine=RiskEngineConfig(bypass=True),
        run_analysis=False,
    ),
)

engine.add_venue(
    venue=venue,
    oms_type=OmsType.NETTING,
    account_type=AccountType.MARGIN,
    starting_balances=[Money(STARTING_USD, USD)],
    base_currency=USD,
    default_leverage=Decimal(1),
)
engine.add_instrument(instrument)
engine.add_data(bars_bid)   # BID side feeds book + drives strategy
engine.add_data(bars_ask)   # ASK side feeds book so BUY orders can cross

# Use the inline subclass — entries submit passive LIMIT orders that
# rest until the market revisits the limit price.  Exits remain market.
engine.add_strategy(
    EMACrossLimitStrategy(
        EMACrossLimitConfig(
            instrument_id=instrument.id,
            bar_type=bar_type_5m_strategy,
            fast_ema_period=FAST_EMA,
            slow_ema_period=SLOW_EMA,
            trade_size=TRADE_SIZE,
            limit_offset_ticks=LIMIT_OFFSET_TICKS,
        ),
    ),
)
print(f"Engine ready.  Strategy: EMACrossLimitStrategy  limit_offset_ticks={LIMIT_OFFSET_TICKS}")

Engine ready.  Strategy: EMACrossLimitStrategy  limit_offset_ticks=50


## 6. Run the backtest

In [7]:
engine.run()
print("Backtest finished.")

Backtest finished.


## 7. Build the order- and position-lifecycle reports

**Order lifecycle.** Iterate every `Order` in `engine.cache.orders()` and walk `order.events` — a chronological list of every `OrderEvent` Nautilus appended. Each event carries two raw timestamps that we render in both **UTC** and **IST (Asia/Kolkata, GMT+0530)** — the timezone the source CSVs are stamped in.

Columns:
- **`ts_event_utc` / `ts_event_ist`** — when the event occurred (venue / matching-engine time in backtest)
- **`ts_init_utc`  / `ts_init_ist`**  — when the event object was constructed (our side)
- **`reason`** — only populated on `OrderRejected` / `OrderDenied` / `OrderCanceled` — useful for diagnosing why fills don't happen

> **Why two `OrderFilled` rows per market order?** Nautilus's matching engine maintains an L1 book whose depth at each price level is capped by the bar's traded volume. A 100k-unit market order typically clears the top level (e.g. 2 units at one price) then walks one tick deeper for the rest (99,998 units). Each match emits its own `OrderFilled` event with its own `last_px` / `last_qty`. If you want **one row per order** with the weighted-average fill price, use `order_fills_report.csv` (`avg_px` column) — the lifecycle report deliberately keeps every individual fill so you can audit slippage.

**Position lifecycle.** A parallel report built from `engine.cache.positions()` — one row per `PositionOpened`, plus one row per `PositionClosed` (if the position has closed). Carries `avg_px_open`, `avg_px_close`, `realized_pnl`, and `duration_seconds` on close. Link to the order CSV via `opening_order_id` ↔ `client_order_id` and `closing_order_id` ↔ `client_order_id`, so you can trace a full trade:

```
order O-...-1   submit / fill  → opens position P-...
position P-...  PositionOpened
... time passes ...
order O-...-2   submit / fill  → closes position P-...
position P-...  PositionClosed  (with realized_pnl, duration_seconds)
```

In [8]:
IST = "Asia/Kolkata"  # source CSVs are stamped GMT+0530


def _fmt(ts_ns: int, tz: str | None = None) -> str:
    """Format a UNIX-ns int as a readable string in UTC or another tz."""
    ts = pd.Timestamp(int(ts_ns), unit="ns", tz="UTC")
    if tz:
        ts = ts.tz_convert(tz)
    return ts.strftime("%Y-%m-%d %H:%M:%S")


def build_lifecycle_report(engine: BacktestEngine) -> pd.DataFrame:
    """One row per OrderEvent across every order in the cache."""
    rows: list[dict] = []
    for order in engine.cache.orders():
        for seq, ev in enumerate(order.events):
            rows.append({
                "client_order_id":  str(order.client_order_id),
                "event_seq":        seq,
                "event_type":       type(ev).__name__,
                "ts_event_utc":     _fmt(ev.ts_event),
                "ts_event_ist":     _fmt(ev.ts_event, IST),
                "ts_init_utc":      _fmt(ev.ts_init),
                "ts_init_ist":      _fmt(ev.ts_init,  IST),
                "order_side":       "BUY" if str(order.side) == "1" else "SELL",
                "order_type":       str(order.order_type),
                "quantity":         str(order.quantity),
                "last_px":          str(getattr(ev, "last_px", "")) or None,
                "last_qty":         str(getattr(ev, "last_qty", "")) or None,
                "venue_order_id":   str(order.venue_order_id) if order.venue_order_id else None,
                "reason":           getattr(ev, "reason", None),
                "instrument_id":    str(order.instrument_id),
                "strategy_id":      str(order.strategy_id),
            })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["client_order_id", "ts_event_utc", "event_seq"])
    return df


def build_position_lifecycle_report(engine: BacktestEngine) -> pd.DataFrame:
    """One row per position-state transition (Opened / Closed) for every
    flip, including snapshots from prior NETTING-OMS flips."""
    rows: list[dict] = []
    all_positions = list(engine.cache.positions()) + list(engine.cache.position_snapshots())

    for pos in all_positions:
        entry_side = "BUY" if str(pos.entry) == "1" else "SELL"
        position_side_open = "LONG" if entry_side == "BUY" else "SHORT"

        rows.append({
            "position_id":      str(pos.id),
            "opening_order_id": str(pos.opening_order_id),
            "event_seq":        0,
            "event_type":       "PositionOpened",
            "ts_event_utc":     _fmt(pos.ts_opened),
            "ts_event_ist":     _fmt(pos.ts_opened, IST),
            "position_side":    position_side_open,
            "entry_side":       entry_side,
            "quantity":         str(pos.peak_qty),
            "avg_px_open":      f"{pos.avg_px_open:.5f}",
            "avg_px_close":     None,
            "realized_pnl":     None,
            "duration_seconds": None,
            "closing_order_id": None,
            "instrument_id":    str(pos.instrument_id),
            "strategy_id":      str(pos.strategy_id),
        })

        if pos.is_closed:
            rows.append({
                "position_id":      str(pos.id),
                "opening_order_id": str(pos.opening_order_id),
                "event_seq":        1,
                "event_type":       "PositionClosed",
                "ts_event_utc":     _fmt(pos.ts_closed),
                "ts_event_ist":     _fmt(pos.ts_closed, IST),
                "position_side":    "FLAT",
                "entry_side":       entry_side,
                "quantity":         str(pos.peak_qty),
                "avg_px_open":      f"{pos.avg_px_open:.5f}",
                "avg_px_close":     f"{pos.avg_px_close:.5f}",
                "realized_pnl":     str(pos.realized_pnl) if pos.realized_pnl is not None else None,
                "duration_seconds": pos.duration_ns / 1e9,
                "closing_order_id": str(pos.closing_order_id) if pos.closing_order_id else None,
                "instrument_id":    str(pos.instrument_id),
                "strategy_id":      str(pos.strategy_id),
            })

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["ts_event_utc", "opening_order_id", "event_seq"])
    return df


lifecycle_df = build_lifecycle_report(engine)
position_lifecycle_df = build_position_lifecycle_report(engine)

n_orders = lifecycle_df["client_order_id"].nunique() if not lifecycle_df.empty else 0
n_trades = position_lifecycle_df["opening_order_id"].nunique() if not position_lifecycle_df.empty else 0
print(f"Captured {len(lifecycle_df):,} order events across {n_orders} orders")
print(f"Captured {len(position_lifecycle_df):,} position events across {n_trades} trades (distinct opening_order_id)")
print("\nOrder event type breakdown:")
print(lifecycle_df["event_type"].value_counts() if not lifecycle_df.empty else "(empty)")
print("\nPosition event type breakdown:")
print(position_lifecycle_df["event_type"].value_counts() if not position_lifecycle_df.empty else "(empty)")

# ── Proof-of-fill-timing ─────────────────────────────────────────────────────
# For each order, compare ts_event_ist of its FIRST OrderSubmitted vs its
# FIRST OrderFilled. With market orders these are identical; with passive
# limit orders the fill happens on a later bar.
if not lifecycle_df.empty:
    sub = (lifecycle_df[lifecycle_df["event_type"] == "OrderSubmitted"]
           .groupby("client_order_id")["ts_event_ist"].first()
           .rename("submit_ist"))
    fill = (lifecycle_df[lifecycle_df["event_type"] == "OrderFilled"]
            .groupby("client_order_id")["ts_event_ist"].first()
            .rename("fill_ist"))
    accept = (lifecycle_df[lifecycle_df["event_type"] == "OrderAccepted"]
              .groupby("client_order_id")["ts_event_ist"].first()
              .rename("accept_ist"))
    timing = pd.concat([sub, accept, fill], axis=1).dropna(subset=["fill_ist"])
    delayed = timing[timing["submit_ist"] != timing["fill_ist"]]
    print(
        f"\nOrders filled on a LATER bar than they were submitted: "
        f"{len(delayed):,} of {len(timing):,} filled orders"
    )
    if len(delayed) > 0:
        print("\nFirst 8 such orders (IST timestamps):")
        cols = [c for c in ["submit_ist", "accept_ist", "fill_ist"] if c in delayed.columns]
        print(delayed[cols].head(8))

Captured 35,423 order events across 7099 orders
Captured 160 position events across 80 trades (distinct opening_order_id)

Order event type breakdown:
event_type
OrderInitialized      7099
OrderSubmitted        7099
OrderAccepted         7019
OrderPendingCancel    6945
OrderCanceled         6945
OrderFilled            316
Name: count, dtype: int64

Position event type breakdown:
event_type
PositionOpened    80
PositionClosed    80
Name: count, dtype: int64

Orders filled on a LATER bar than they were submitted: 80 of 160 filled orders

First 8 such orders (IST timestamps):
                                            submit_ist           accept_ist  \
client_order_id                                                               
O-20240401-141000-APR2024-000-188  2024-04-01 19:40:00  2024-04-01 19:40:00   
O-20240402-075500-APR2024-000-365  2024-04-02 13:25:00  2024-04-02 13:25:00   
O-20240402-144500-APR2024-000-442  2024-04-02 20:15:00  2024-04-02 20:15:00   
O-20240403-085500-APR2024

## 8. Pull the three built-in reports

In [9]:
trader = engine.trader

fills_df     = trader.generate_order_fills_report()
positions_df = trader.generate_positions_report()
account_df   = trader.generate_account_report(venue)

print(f"fills_df:     {fills_df.shape}")
print(f"positions_df: {positions_df.shape}")
print(f"account_df:   {account_df.shape}")
fills_df.head()

fills_df:     (160, 36)
positions_df: (80, 21)
account_df:   (14281, 10)


,trader_id,strategy_id,instrument_id,venue_order_id,position_id,account_id,last_trade_id,type,side,quantity,...,order_list_id,linked_order_ids,parent_order_id,exec_algorithm_id,exec_algorithm_params,exec_spawn_id,tags,init_id,ts_init,ts_last
client_order_id,,,,,,,,,,,,,,,,,,,,,
O-20240401-141000-APR2024-000-188,EMA-CROSS-APR2024,EMACrossLimitStrategy-000,EURUSD.FOREX_MS,FOREX_MS-1-188,EURUSD.FOREX_MS-EMACrossLimitStrategy-000,FOREX_MS-001,FOREX_MS-1-002,LIMIT,SELL,100000,...,None,None,None,None,None,None,[EMA Cross LIMIT SELL: fast(10)=1.07651 slow(...,fce8577c-305a-43c8-b3b4-a0b33e7dd1a3,2024-04-01 14:10:00+00:00,2024-04-01 14:15:00+00:00
O-20240401-172000-APR2024-000-189,EMA-CROSS-APR2024,EMACrossLimitStrategy-000,EURUSD.FOREX_MS,FOREX_MS-1-189,EURUSD.FOREX_MS-EMACrossLimitStrategy-000,FOREX_MS-001,FOREX_MS-1-004,MARKET,BUY,100000,...,None,None,None,None,None,None,None,5f35736c-ec75-4a15-b4aa-517413fd4690,2024-04-01 17:20:00+00:00,2024-04-01 17:20:00+00:00
O-20240402-075500-APR2024-000-365,EMA-CROSS-APR2024,EMACrossLimitStrategy-000,EURUSD.FOREX_MS,FOREX_MS-1-365,EURUSD.FOREX_MS-EMACrossLimitStrategy-000,FOREX_MS-001,FOREX_MS-1-006,LIMIT,BUY,100000,...,None,None,None,None,None,None,[EMA Cross LIMIT BUY: fast(10)=1.07352 slow(2...,646e6c7c-9183-4d0b-9d75-fc522eb73202,2024-04-02 07:55:00+00:00,2024-04-02 08:00:00+00:00
O-20240402-083000-APR2024-000-366,EMA-CROSS-APR2024,EMACrossLimitStrategy-000,EURUSD.FOREX_MS,FOREX_MS-1-366,EURUSD.FOREX_MS-EMACrossLimitStrategy-000,FOREX_MS-001,FOREX_MS-1-008,MARKET,SELL,100000,...,None,None,None,None,None,None,None,170684ee-d34a-46bd-8398-30dec2d1d2d1,2024-04-02 08:30:00+00:00,2024-04-02 08:30:00+00:00
O-20240402-144500-APR2024-000-442,EMA-CROSS-APR2024,EMACrossLimitStrategy-000,EURUSD.FOREX_MS,FOREX_MS-1-442,EURUSD.FOREX_MS-EMACrossLimitStrategy-000,FOREX_MS-001,FOREX_MS-1-010,LIMIT,BUY,100000,...,None,None,None,None,None,None,[EMA Cross LIMIT BUY: fast(10)=1.07715 slow(2...,cc7fdff6-05a5-4de1-a868-2859932c5952,2024-04-02 14:45:00+00:00,2024-04-02 14:50:00+00:00


## 9. Write all four CSVs

In [10]:
fills_df.to_csv(OUT_DIR / "order_fills_report.csv")
positions_df.to_csv(OUT_DIR / "positions_report.csv")
account_df.to_csv(OUT_DIR / "account_report.csv")
lifecycle_df.to_csv(OUT_DIR / "order_lifecycle_report.csv", index=False)
position_lifecycle_df.to_csv(OUT_DIR / "position_lifecycle_report.csv", index=False)

for f in sorted(OUT_DIR.glob("*.csv")):
    print(f"  {f.name:35s}  {f.stat().st_size:>12,} bytes")
print(f"\nAll reports written to: {OUT_DIR}")

  account_report.csv                      2,287,191 bytes
  order_fills_report.csv                     69,023 bytes
  order_lifecycle_report.csv              7,375,041 bytes
  position_lifecycle_report.csv              43,526 bytes
  positions_report.csv                       33,173 bytes

All reports written to: d:\mcube_test_version_1_updated\m-cube_version1\reports\ema_cross_april2024


## 10. Spot-check: full lifecycle of a single order

Pick the first order and show every event timestamp side by side. On a happy path you'll see `OrderInitialized → OrderSubmitted → OrderAccepted → OrderFilled` with `ts_event` monotonically non-decreasing and `ts_init` ≥ `ts_event`. If anything rejected, the `reason` column tells you why.

In [11]:
if not lifecycle_df.empty:
    first_id = lifecycle_df["client_order_id"].iloc[0]
    print(f"Lifecycle for first order {first_id}:\n")
    one = lifecycle_df[lifecycle_df["client_order_id"] == first_id]
    order_cols = [
        "event_seq", "event_type",
        "ts_event_utc", "ts_event_ist",
        "venue_order_id", "order_side", "last_px", "last_qty", "reason",
    ]
    display(one[order_cols])
else:
    print("No orders were created — strategy never fired. Check EMA periods vs warmup.")

if not position_lifecycle_df.empty:
    first_pid = position_lifecycle_df["position_id"].iloc[0]
    print(f"\nLifecycle for first position {first_pid}:\n")
    one_pos = position_lifecycle_df[position_lifecycle_df["position_id"] == first_pid]
    position_cols = [
        "event_seq", "event_type",
        "ts_event_utc", "ts_event_ist",
        "entry_side", "quantity", "avg_px_open", "avg_px_close",
        "realized_pnl", "duration_seconds",
        "opening_order_id", "closing_order_id",
    ]
    display(one_pos[position_cols])
else:
    print("\nNo positions were created.")

Lifecycle for first order O-20240331-223500-APR2024-000-1:



,event_seq,event_type,ts_event_utc,ts_event_ist,venue_order_id,order_side,last_px,last_qty,reason
0,0,OrderInitialized,2024-03-31 22:35:00,2024-04-01 04:05:00,FOREX_MS-1-001,BUY,None,None,None
1,1,OrderSubmitted,2024-03-31 22:35:00,2024-04-01 04:05:00,FOREX_MS-1-001,BUY,None,None,None
2,2,OrderAccepted,2024-03-31 22:35:00,2024-04-01 04:05:00,FOREX_MS-1-001,BUY,None,None,None
3,3,OrderPendingCancel,2024-03-31 22:40:00,2024-04-01 04:10:00,FOREX_MS-1-001,BUY,None,None,None
4,4,OrderCanceled,2024-03-31 22:40:00,2024-04-01 04:10:00,FOREX_MS-1-001,BUY,None,None,None



Lifecycle for first position EURUSD.FOREX_MS-EMACrossLimitStrategy-000-57f9e6f0-a08e-465a-a180-b447a5c7f013:



,event_seq,event_type,ts_event_utc,ts_event_ist,entry_side,quantity,avg_px_open,avg_px_close,realized_pnl,duration_seconds,opening_order_id,closing_order_id
2,0,PositionOpened,2024-04-01 14:15:00,2024-04-01 19:45:00,SELL,100000,1.07507,None,None,NaN,O-20240401-141000-APR2024-000-188,None
3,1,PositionClosed,2024-04-01 17:20:00,2024-04-01 22:50:00,SELL,100000,1.07507,1.07431,-138.94 USD,11100.0,O-20240401-141000-APR2024-000-188,O-20240401-172000-APR2024-000-189


## 11. Cleanup

In [12]:
engine.dispose()
print("Engine disposed.")

Engine disposed.
